In [ ]:
import io
import logging

logging.basicConfig(level=logging.ERROR)

In [ ]:
def process_stream(file_stream):
    # transforms raw text stream into dicts
    # uses next() to extract schema headers

    try:
        # use next() to grab the single first row, advancing the stream's pointer so the subsequent loop skips the header
        raw_header = next(file_stream)

        # tokenize headers to use as dict keys
        headers = [col.strip() for col in raw_header.split(",")]

    except StopIteration:
        print("Data stream is empty")
        return

    for line in file_stream:
        cleaned_line = line.strip()
        if not cleaned_line:
            continue

        try:
            print(cleaned_line)
            values = [val.strip() for val in cleaned_line.split(",")]
            if len(values) != len(headers):
                raise ValueError("Row count mismatch")

            record = dict(zip(headers, values))
            record["price"] = int(record["price"])

            yield record

        except ValueError as e:
            continue

if __name__ == "__main__":
    mock_csv_file = io.StringIO("""prop_id, city, price
        p101, Seattle, 850000
        p102, Bellevue, 1200000
        p103, Tacoma, 550000
    """)

    stream_handle = (line for line in mock_csv_file)
    pipeline_output = process_stream(stream_handle)

    for rec in pipeline_output:
        print(rec)